# 🏦 Banking Credit Card Fraud — PySpark EDA

> **Dataset:** Credit Card Fraud Detection (Kaggle)  
> **Source:** https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud  
> **Size:** 284,807 transactions | 31 columns | ~143 MB  
> **Domain:** Banking & Finance  

## Project Overview

This notebook performs a complete end-to-end PySpark EDA on real-world credit card transaction data. The dataset contains transactions made by European cardholders over 2 days in September 2013.

**Key challenge:** The dataset is **highly imbalanced** — only 492 frauds out of 284,807 transactions (0.172%).

## Pipeline
1. Setup & SparkSession
2. Data Ingestion
3. Data Cleaning
4. Exploratory Data Analysis (EDA)
5. Feature Engineering
6. Spark SQL Analysis
7. Performance Optimization
8. Business Insights

---
## 📦 Section 1: Setup & SparkSession Configuration

In [ ]:
# ── Standard Imports ─────────────────────────────────────────
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.append('..')

# Data
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# PySpark
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Project modules
from src.utils.spark_session import create_spark_session
from src.utils.helpers import profile_dataframe, null_percentage_report

print('✅ All imports successful')

In [ ]:
# ── Create SparkSession ───────────────────────────────────────
# Config is loaded from config/spark_config.yaml
spark = create_spark_session(config_path='../config/spark_config.yaml')

print(f'Spark Version : {spark.version}')
print(f'Python Version: {sys.version}')
print(f'Pandas Version: {pd.__version__}')
print(f'\nSpark UI: http://localhost:4040')

---
## 📥 Section 2: Data Ingestion

**Dataset Download:** https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

Place `creditcard.csv` into `data/raw/` before running this cell.

In [ ]:
from src.ingestion.data_loader import load_csv, save_as_parquet, load_parquet

RAW_CSV     = '../data/raw/creditcard.csv'
PARQUET_OUT = '../data/processed/creditcard_clean.parquet'

# ── Load raw CSV with explicit schema ───────────────────────
# Using explicit schema avoids a full extra scan for type inference
df_raw = load_csv(spark, RAW_CSV)

print(f'Rows   : {df_raw.count():,}')
print(f'Columns: {len(df_raw.columns)}')
df_raw.printSchema()

In [ ]:
# ── Preview raw data ─────────────────────────────────────────
df_raw.show(5, truncate=True)

In [ ]:
# ── Basic descriptive statistics ─────────────────────────────
# PySpark describe() returns: count, mean, stddev, min, max
df_raw.select('Time', 'Amount', 'Class').describe().show()

---
## 🧹 Section 3: Data Cleaning

In [ ]:
from src.cleaning.data_cleaner import run_cleaning_pipeline
from src.utils.helpers import null_percentage_report, count_duplicates

# ── Check nulls before cleaning ──────────────────────────────
print('=== NULL ANALYSIS (Before Cleaning) ===')
null_report = null_percentage_report(df_raw)
print(null_report.head(10))

# ── Check duplicates ─────────────────────────────────────────
print('\n=== DUPLICATE CHECK ===')
count_duplicates(df_raw)

In [ ]:
# ── Run full cleaning pipeline ───────────────────────────────
df_clean = run_cleaning_pipeline(df_raw)

print(f'Rows after cleaning: {df_clean.count():,}')
df_clean.select('Amount', 'Class').describe().show()

In [ ]:
# ── Save cleaned data as Parquet ─────────────────────────────
# Parquet is 3-10x faster for subsequent reads
save_as_parquet(df_clean, PARQUET_OUT)
print(f'Saved to: {PARQUET_OUT}')

---
## 🔍 Section 4: Exploratory Data Analysis (EDA)

In [ ]:
from src.eda.exploratory_analysis import (
    analyze_class_distribution,
    analyze_transaction_amounts,
    analyze_time_patterns,
    analyze_correlations,
    plot_top_features_boxplot,
    fraud_heatmap_hourly
)

# 4.1 Class Distribution
class_dist = analyze_class_distribution(df_clean)

In [ ]:
# 4.2 Transaction Amount Analysis
analyze_transaction_amounts(df_clean)

In [ ]:
# 4.3 Time Patterns
analyze_time_patterns(df_clean)

In [ ]:
# 4.4 Feature Correlations with Fraud
corr_df = analyze_correlations(df_clean, top_features=10)

In [ ]:
# 4.5 Box Plots for Top V-Features
top_features = corr_df.head(8)['feature'].tolist()
plot_top_features_boxplot(df_clean, top_features)

In [ ]:
# 4.6 Fraud Heatmap: Hour × Amount Band
fraud_heatmap_hourly(df_clean)

---
## ⚙️ Section 5: Feature Engineering

In [ ]:
from src.feature_engineering.feature_builder import run_feature_engineering

# Run full feature engineering pipeline
df_features = run_feature_engineering(df_clean)

# Preview new features
df_features.select(
    'Time', 'Amount', 'Class',
    'Hour', 'DayPeriod', 'IsNight',
    'LogAmount', 'AmountBin', 'IsLargeAmt',
    'RollingAvgAmount', 'RollingMaxAmount'
).show(10, truncate=False)

In [ ]:
# ── Fraud rate by DayPeriod ───────────────────────────────────
df_features.groupBy('DayPeriod') \
    .agg(
        F.count('*').alias('total'),
        F.sum('Class').alias('fraud'),
        F.round(F.mean('Class') * 100, 4).alias('fraud_rate_pct')
    ) \
    .orderBy('fraud_rate_pct', ascending=False) \
    .show()

---
## 🗃️ Section 6: Spark SQL Analysis

In [ ]:
from src.sql_analysis.spark_sql_queries import (
    register_temp_view,
    run_all_sql_analysis
)

# Register as temp view for SQL access
register_temp_view(df_features, 'transactions')

# Verify table is accessible
spark.sql('SELECT COUNT(*) AS row_count FROM transactions').show()

In [ ]:
# ── Run all SQL queries ───────────────────────────────────────
sql_results = run_all_sql_analysis(spark)

In [ ]:
# ── Custom Spark SQL query ────────────────────────────────────
# You can write any SQL here and Spark compiles it to an optimized plan

spark.sql("""
    SELECT
        DayPeriod,
        AmountBin,
        COUNT(*)                AS txn_count,
        SUM(Class)              AS fraud_count,
        ROUND(AVG(Amount), 2)   AS avg_amount,
        ROUND(SUM(Class) * 100.0 / COUNT(*), 4) AS fraud_rate_pct
    FROM transactions
    GROUP BY DayPeriod, AmountBin
    ORDER BY fraud_rate_pct DESC
""").show(20, truncate=False)

---
## ⚡ Section 7: Performance Optimization

### 7.1 Caching
Cache DataFrames that are reused multiple times to avoid re-computation.

In [ ]:
# ── Caching Demo ──────────────────────────────────────────────

# Without cache: every action re-reads from disk and re-applies transforms
# With cache:    first action materializes in memory; subsequent actions use it

import time

# First pass WITHOUT cache
t0 = time.time()
df_features.filter(F.col('Class') == 1).count()
t_no_cache = time.time() - t0
print(f'Without cache: {t_no_cache:.2f}s')

# Cache the DataFrame
df_features.cache()          # lazy — materialised on first action
df_features.count()           # trigger materialisation

# Second pass WITH cache (should be ~2-5x faster)
t0 = time.time()
df_features.filter(F.col('Class') == 1).count()
t_cached = time.time() - t0
print(f'With cache   : {t_cached:.2f}s')
print(f'Speedup      : {t_no_cache / max(t_cached, 0.001):.1f}x')

In [ ]:
# ── Partitioning Demo ─────────────────────────────────────────

# Check current partition count
print(f'Current partitions: {df_features.rdd.getNumPartitions()}')

# Repartition: increase parallelism for large operations
df_repartitioned = df_features.repartition(8, F.col('IsNight'))
print(f'After repartition : {df_repartitioned.rdd.getNumPartitions()}')

# Coalesce: reduce partitions after filtering (no full shuffle)
df_fraud_only = df_features.filter(F.col('Class') == 1).coalesce(2)
print(f'Fraud-only partitions: {df_fraud_only.rdd.getNumPartitions()}')

In [ ]:
# ── Explain Plan ──────────────────────────────────────────────
# View Spark's query execution plan (Physical Plan)
df_features \
    .filter(F.col('Amount') > 100) \
    .groupBy('Class') \
    .agg(F.mean('Amount')) \
    .explain()   # Pass extended=True for full logical + physical plan

---
## 💡 Section 8: Business Insights

Summarising actionable findings from the EDA:

In [ ]:
# ── Business Insight 1: Fraud rate is tiny but high-value ─────
spark.sql("""
    SELECT
        ROUND(SUM(CASE WHEN Class=1 THEN Amount ELSE 0 END), 2) AS fraud_total_usd,
        ROUND(SUM(Amount), 2)                                   AS all_txn_total_usd,
        ROUND(
            SUM(CASE WHEN Class=1 THEN Amount ELSE 0 END)
            / SUM(Amount) * 100, 4
        )                                                       AS fraud_amount_pct
    FROM transactions
""").show()

In [ ]:
# ── Business Insight 2: Night has higher fraud rate ───────────
spark.sql("""
    SELECT
        IsNight,
        COUNT(*)                 AS total,
        SUM(Class)               AS fraud,
        ROUND(SUM(Class)*100.0 / COUNT(*), 4) AS fraud_rate_pct
    FROM transactions
    GROUP BY IsNight
    ORDER BY IsNight
""").show()

In [ ]:
# ── Business Insight 3: Fraud amount is typically small ───────
spark.sql("""
    SELECT
        Class,
        ROUND(percentile_approx(Amount, 0.50), 2) AS median_amount,
        ROUND(AVG(Amount), 2)                     AS mean_amount,
        ROUND(MAX(Amount), 2)                     AS max_amount
    FROM transactions
    GROUP BY Class
""").show()

In [ ]:
# ── Cleanup: unpersist cache and stop Spark ───────────────────
df_features.unpersist()

# Optionally stop session (comment out if running more cells)
# spark.stop()
# print('Spark session stopped.')

print('\n✅ Notebook execution complete!')
print('   All plots saved to: outputs/plots/')

---
## 📋 Summary of Findings

| Insight | Finding | Business Action |
|---|---|---|
| Class imbalance | 0.17% fraud | Use SMOTE / class weighting in ML |
| Fraud amounts | Median ~$22, smaller than legit | Flag micro-transactions from new sources |
| Time patterns | Night-time peak | Enhanced auth between 10pm–6am |
| Top features | V14, V17, V12, V10 strongest signal | Use these in fraud scoring model |
| Financial risk | Fraud = ~0.27% of total $ | Focus on high-value fraud cases |